In [ ]:
#| default_exp mcp

## MCP server

Expose nbskill notebook operations as native MCP tools. This is the preferred integration for careful single-notebook reads and edits because multiline notebook cells travel as structured tool arguments rather than shell-quoted strings. Keep MCP calls serial; use the CLI through uv run for batch operations and final verification.

The command-line functions are useful on their own, but coding agents work best when the same operations are available as structured tools. This notebook exposes the project through a FastMCP server while keeping the server layer thin and predictable.

The MCP server should stay boring on purpose. Each tool accepts structured arguments, captures printed output, uses notebook locks where file operations can collide, and delegates the actual work to the same functions tested elsewhere.

```python
mcp = create_mcp()
# MCP clients see tools such as read_nb, write_nb, update_cell, exec_nb, and diff_nb.
```

In [0]:
from nbskill.mcp import capture_call, create_mcp

def _demo_tool():
    print("captured output")

print(capture_call(_demo_tool))
print(type(create_mcp()).__name__)

captured output
FastMCP


In [ ]:
#| export
import os
import sys
from contextlib import redirect_stdout, redirect_stderr
from importlib.metadata import PackageNotFoundError, version
from io import StringIO
from pathlib import Path

from fastcore.script import Param, call_parse
from fastmcp import FastMCP
from fastmcp.tools import ToolResult
from mcp.types import TextContent

from nbskill.convert import py2nb as _py2nb
from nbskill.convert import py2nbs as _py2nbs
from nbskill.edit_interactive import execute_plan as _execute_plan
from nbskill.execute import exec_nb as _exec_nb
from nbskill.graph import private_symbol_report as _private_symbol_report
from nbskill.graph import symbol_graph as _symbol_graph
from nbskill.parallel import notebook_locks
from nbskill.read import read_nb as _read_nb
from nbskill.read import show_doc as _show_doc
from nbskill.review import style_check as _style_check
from nbskill.review import diff_nb as _diff_nb
from nbskill.write import apply_nb as _apply_nb
from nbskill.write import update_cell as _update_cell
from nbskill.write import write_nb as _write_nb

### Capturing command output

The MCP tools should return text, not leak stdout and stderr into the server process. These helpers capture each underlying function call and convert its visible result into one response string.

In [ ]:
#| export
def as_text(value):
    return "" if value is None else str(value)


def _package_version(name="nbskill"):
    try: return version(name)
    except PackageNotFoundError: return "unknown"


def capture_call(func, **kwargs):
    out, err = StringIO(), StringIO()
    with redirect_stdout(out), redirect_stderr(err):
        result = func(**kwargs)
    chunks = []
    if out.getvalue(): chunks.append(out.getvalue().rstrip())
    if err.getvalue(): chunks.append(err.getvalue().rstrip())
    if result is not None and not chunks: chunks.append(as_text(result))
    return chr(10).join(chunk for chunk in chunks if chunk)


def capture_notebook_call(func, *paths, **kwargs):
    "Capture a call while holding per-notebook locks for `paths`."
    with notebook_locks(*paths):
        return capture_call(func, **kwargs)

### Registering notebook tools

`create_mcp` is the bridge between this package and an agent client. Each tool is a thin wrapper around a public function, with notebook locks around operations that touch shared files.

In [ ]:
#| export
def create_mcp():
    "Create the nbskill FastMCP server."
    capabilities = (
        "read_nb,show_doc,write_nb,update_cell,exec_nb,diff_nb,execute_plan,"
        "symbol_graph,private_symbol_report,apply_nb,style_check,py2nb,py2nbs"
    )
    mcp = FastMCP(
        "nbskill",
        instructions=(
            "Work notebook-first in nbdev projects. Prefer read_nb/show_doc for context, "
            "write_nb/update_cell for edits, exec_nb for visible notebook execution, "
            "and execute_plan for Lisette-powered single-notebook plan execution. "
            "Notebook operations are concurrency-safe: calls touching the same notebook are serialized, "
            "calls touching different notebooks can run in parallel, and execution uses a global semaphore. "
            "Keep documentation before exported code and show-off examples after it."
        ),
    )

    @mcp.tool(name="healthcheck")
    def healthcheck_tool() -> ToolResult:
        "Return a small status report for the local nbskill MCP server."
        full_output = "\n".join([
            "nbskill mcp ok",
            f"version={_package_version()}",
            f"cwd={Path.cwd()}",
            f"python={sys.executable}",
            f"pid={os.getpid()}",
            f"capabilities={capabilities}",
            "parallel=same-notebook operations serialized; different notebooks may run in parallel",
            "execution=global semaphore with one active notebook execution",
            "schema_refresh=restart or reconnect the MCP client after reinstall/export to refresh tool schemas",
        ])
        summary = f"healthcheck completed\n\n{full_output}"
        return ToolResult(
            content=[TextContent(type="text", text=summary)],
            structured_content={"summary": summary, "full_output": full_output},
        )

    @mcp.tool(name="read_nb")
    def read_nb_tool(
        path: str,
        query: str | None = None,
        cell_id: str | None = None,
        chapter: str | None = None,
        cell_type: str | None = None,
        contains: str | None = None,
        context: str = "overview",
        show_ids: bool = False,
    ) -> str:
        "Read compact notebook views; context controls overview, precise source, or full surrounding docs/examples."
        return capture_notebook_call(
            _read_nb, path, path=path, query=query, cell_id=cell_id, chapter=chapter,
            cell_type=cell_type, contains=contains, context=context, show_ids=show_ids,
        )

    @mcp.tool(name="show_doc")
    def show_doc_tool(path: str, symbol: str, context: int = 2, source: bool = False, show_ids: bool = False) -> str:
        "Show rationale/docs, exported code, and show-off examples for a symbol."
        return capture_notebook_call(_show_doc, path, path=path, symbol=symbol, context=context, source=source, show_ids=show_ids)

    @mcp.tool(name="write_nb")
    def write_nb_tool(
        path: str,
        cells: str = "",
        before_id: str | None = None,
        after_id: str | None = None,
        chapter: str | None = None,
        replace: bool = False,
        cell_type: str = "code",
        export: bool = True,
        run_test: bool = False,
        run_style: bool = False,
        style_strict: bool = False,
        validate_code: bool = True,
        old_str: str | None = None,
        new_str: str | None = None,
        dry_run: bool = False,
        show_cells: bool = False,
    ) -> str:
        "Write cells to a notebook, or replace literal text across notebooks."
        return capture_notebook_call(
            _write_nb, path, path=path, cells=cells, cells_file=None, before_id=before_id, after_id=after_id,
            chapter=chapter, replace=replace, cell_type=cell_type, export=export, run_test=run_test,
            run_style=run_style, style_strict=style_strict, validate_code=validate_code,
            old_str=old_str, new_str=new_str, dry_run=dry_run, show_cells=show_cells,
        )

    @mcp.tool(name="update_cell")
    def update_cell_tool(
        path: str,
        new: str = "",
        cell_id: str | None = None,
        old_str: str | None = None,
        line_range: str | None = None,
        source_hash: str | None = None,
        cell_type: str = "code",
        export: bool = True,
        run_test: bool = False,
        validate_code: bool = True,
        dry_run: bool = False,
    ) -> str:
        "Update a cell, replace old_str, or replace 1-based inclusive line_range."
        return capture_notebook_call(
            _update_cell, path, path=path, new=new, new_file=None, cell_id=cell_id, old_str=old_str,
            line_range=line_range, source_hash=source_hash, cell_type=cell_type, export=export,
            run_test=run_test, validate_code=validate_code, dry_run=dry_run,
        )

    @mcp.tool(name="exec_nb")
    def exec_nb_tool(
        path: str,
        dest: str | None = None,
        exc_stop: bool = False,
        up2id: int | str | None = None,
        chapter: str | None = None,
        timeout: int = 30,
        show_output: bool = True,
        verbose: bool = False,
    ) -> str:
        "Execute a notebook and return visible outputs/errors, with a per-cell timeout."
        return capture_notebook_call(
            _exec_nb, path, dest or path, path=path, dest=dest, exc_stop=exc_stop, up2id=up2id,
            chapter=chapter, timeout=timeout, show_output=show_output, verbose=verbose,
        )

    @mcp.tool(name="diff_nb")
    def diff_nb_tool(path: str, ref_a: str | None = "HEAD", ref_b: str | None = None, adds: bool = True, changes: bool = True, dels: bool = False) -> str:
        "Diff code cells only. nbskill metadata-only changes are summarized, not expanded."
        return capture_notebook_call(_diff_nb, path, path=path, ref_a=ref_a, ref_b=ref_b, adds=adds, changes=changes, dels=dels)

    @mcp.tool(name="execute_plan")
    def execute_plan_tool(
        notebook: str,
        plan: str,
        model: str | None = None,
        max_steps: int = 20,
        timeout: int = 30,
        export: bool = True,
    ) -> str:
        "Run a Lisette edit-interactive loop against one notebook."
        return capture_call(
            _execute_plan, notebook=notebook, plan=plan, model=model,
            max_steps=max_steps, timeout=timeout, export=export,
        )


    @mcp.tool(name="symbol_graph")
    def symbol_graph_tool(path: str = "nbs", symbol: str = "") -> str:
        "Show definitions, callers, and callees for a notebook symbol."
        return capture_call(_symbol_graph, path=path, symbol=symbol)

    @mcp.tool(name="private_symbol_report")
    def private_symbol_report_tool(path: str = "nbs") -> str:
        "Show cross-notebook calls to private `_` symbols."
        return capture_call(_private_symbol_report, path=path)

    @mcp.tool(name="apply_nb")
    def apply_nb_tool(spec_path: str = "dev/nbskill-op.toml") -> str:
        "Apply the archived TOML manifest workflow. Prefer write_nb/update_cell MCP tools when available."
        return capture_call(_apply_nb, spec_path=spec_path)

    @mcp.tool(name="style_check")
    def style_check_tool(path: str = ".", skip_folder_re: str | None = None, skip_path: str | None = None, strict: bool = False) -> str:
        "Print fast.ai style hints."
        return capture_call(_style_check, path=path, skip_folder_re=skip_folder_re, skip_path=skip_path, strict=strict)

    @mcp.tool(name="py2nb")
    def py2nb_tool(path: str, nbs_path: str = "nbs", dest: str | None = None, class_lines: int = 100, method_lines: int = 10) -> str:
        "Convert a Python file into an nbdev notebook."
        return capture_call(_py2nb, path=path, nbs_path=nbs_path, dest=dest, class_lines=class_lines, method_lines=method_lines)

    @mcp.tool(name="py2nbs")
    def py2nbs_tool(path: str, nbs_path: str = "nbs", recursive: bool = True, maxdepth: int | None = None, preserve_tree: bool = True, class_lines: int = 100, method_lines: int = 10) -> str:
        "Convert Python files in a folder into nbdev notebooks."
        return capture_call(
            _py2nbs, path=path, nbs_path=nbs_path, recursive=recursive, maxdepth=maxdepth,
            preserve_tree=preserve_tree, class_lines=class_lines, method_lines=method_lines,
        )

    return mcp

### Running the server

The CLI entry point only chooses the transport and starts FastMCP. Keeping startup separate from tool registration makes `create_mcp` easy to test without launching a long-running server.

In [ ]:
#| export
@call_parse
def main(
    transport: str = "stdio",  # MCP transport; stdio is what Codex/Claude use for local servers
    show_banner: bool = False,  # Show FastMCP startup banner
):
    "Run the nbskill MCP server."
    create_mcp().run(transport=transport, show_banner=show_banner)

In [ ]:
from nbskill.mcp import create_mcp

mcp = create_mcp()
tools = {tool.name: tool for tool in await mcp.list_tools()}
assert {"healthcheck", "read_nb", "write_nb", "update_cell", "exec_nb", "show_doc", "execute_plan", "symbol_graph", "private_symbol_report"} <= set(tools)
assert "show_cells" in str(tools["write_nb"].parameters)
health = await mcp.call_tool("healthcheck", {})
health_text = str(health)
assert "version=" in health_text
assert "capabilities=" in health_text
assert "schema_refresh=" in health_text

In [ ]:
import nbskill.mcp as _mcp_mod

calls = {}
old_execute_plan = _mcp_mod._execute_plan

try:
    def fake_execute_plan(**kwargs):
        calls.update(kwargs)
        return "delegated"

    _mcp_mod._execute_plan = fake_execute_plan
    mcp = _mcp_mod.create_mcp()
    result = await mcp.call_tool(
        "execute_plan",
        {
            "notebook": "nbs/index.ipynb",
            "plan": "noop",
            "model": "fake",
            "max_steps": 1,
            "timeout": 2,
            "export": False,
        },
    )
    assert calls == {
        "notebook": "nbs/index.ipynb",
        "plan": "noop",
        "model": "fake",
        "max_steps": 1,
        "timeout": 2,
        "export": False,
    }
    assert "delegated" in str(result)
finally:
    _mcp_mod._execute_plan = old_execute_plan

In [ ]:
import tempfile as _tempfile
from pathlib import Path as _Path
from fastcore.nbio import mk_cell, new_nb
from fastcore.nbio import write_nb as _write_nb
from nbskill.mcp import capture_call
from nbskill.read import read_nb

with _tempfile.TemporaryDirectory() as td:
    path = _Path(td) / "sample.ipynb"
    _write_nb(new_nb([mk_cell("#| default_exp sample", cell_type="code")]), path)
    text = capture_call(read_nb, path=str(path), context="overview")
    assert "default_exp sample" in text

In [0]:
from nbskill.mcp import as_text, capture_call

assert as_text(None) == ""
assert as_text({"ok": True}) == "{'ok': True}"
assert capture_call(lambda: "returned") == "returned"

def _prints_and_returns():
    print("printed")
    return "returned"

assert capture_call(_prints_and_returns) == "printed"